# 01 - Pass 1: image-only transcription

Each test photo is transcribed on its own (no question, no reference) with `Qwen/Qwen3-VL-235B-A22B-Instruct`, temperature 0, at most 16,384 new tokens. The prompt is `resources/prompts/pass1_transcribe.txt`.

**Reads** `data/competition_data/test.csv`, `images/test/`.  **Writes** `work/pass1/records/<trace_id>.json`, `work/pass1/transcriptions.csv`.
The run is resumable: finished rows are skipped, failed rows are retried (`retry_rounds` in `configs/models.json`).

**Default: smoke test on 5 rows.** Set `LIMIT = None` in the second cell for the full run.

In [ ]:
import os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'models.json').is_file())
os.chdir(REPO); sys.path.insert(0, str(REPO / 'src'))
from misc_pipeline import common, runner
LIMIT = 5         # SMOKE TEST: first 5 test rows only. Set LIMIT = None for the full 3,498 rows (finished rows are kept and skipped, so the full run resumes from the smoke test).
print('repo:', REPO, '| data:', common.DATA, '| work:', common.WORK)

In [ ]:
from misc_pipeline import pass1
jobs = pass1.build_jobs(LIMIT)
print(len(jobs), 'rows | prompt chars:', len(jobs[0]['prompt']))

In [ ]:
summary = runner.run_stage('pass1', jobs, common.STAGE_DIRS['pass1'], pass1.parse, label_images=False)
print(summary)

In [ ]:
exp = pass1.export(jobs); print(exp)
if exp['complete']:
    rows = common.read_csv(common.STAGE_DIRS['pass1'] / 'transcriptions.csv')
    assert [r['trace_id'] for r in rows] == [j['trace_id'] for j in jobs] and all(r['ocr'].strip() for r in rows)
    print('OK:', len(rows), 'transcriptions ->', common.STAGE_DIRS['pass1'] / 'transcriptions.csv')
    print(rows[0]['ocr'][:600])

## Free the GPUs
The stage already closes its model when it ends. Run this cell after an interrupted run or an error, or before starting the next notebook in the same kernel.

In [ ]:
import gc
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print('GPU memory in use (GB):', [round(torch.cuda.memory_allocated(i) / 1e9, 1) for i in range(torch.cuda.device_count())])
except ImportError:
    pass
# If memory is still held, restart the kernel (Kernel > Restart); vLLM worker processes then exit.

## Optional: delete the downloaded model
The weights stay in the Hugging Face cache (`HF_HOME`) and are re-downloaded on the next use. Set `DELETE_MODEL = True` to remove `Qwen/Qwen3-VL-235B-A22B-Instruct` from the cache now. Keep it if a later notebook uses the same model (notebooks 02 and 03 share one model, as do 04 and 05).

In [ ]:
DELETE_MODEL = False
MODEL = 'Qwen/Qwen3-VL-235B-A22B-Instruct'
if DELETE_MODEL:
    from huggingface_hub import scan_cache_dir
    info = scan_cache_dir()
    hashes = [r.commit_hash for repo in info.repos if repo.repo_id == MODEL for r in repo.revisions]
    if hashes:
        strategy = info.delete_revisions(*hashes); print('freeing %.1f GB' % (strategy.expected_freed_size / 1e9)); strategy.execute()
    else:
        print(MODEL, 'is not in the cache')
else:
    print('model kept in the cache')